In [5]:
import cv2
import numpy as np
import time
from ultralytics import YOLO
from shapely.geometry import Point, Polygon
from collections import deque

In [ ]:
model = YOLO('yolov8s.pt')

# polygon_points = [[5, 306], [978, 311], [1009, 696], [3, 709]]
polygon_points = [[0, 300], [1048, 300], [1048, 800], [0, 800], [0, 300]]
polygon = Polygon(polygon_points)

crowd_threshold = 8
frame_queue = deque()
average_duration = 10
frame_interval = 3

def is_in_polygon(x, y):
    point = Point(x, y)
    return polygon.contains(point)

In [7]:
# specify video path
cap = cv2.VideoCapture("./violence_3.mp4")
frame_count = 0
width = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
fourcc = cv2.VideoWriter_fourcc(*"h264")
# specify saved video path
videowriter = cv2.VideoWriter("./crowd_3.mp4", fourcc, 5.0, (int(width), int(height)))

OpenCV: FFMPEG: tag 0x34363268/'h264' is not supported with codec id 27 and format 'mp4 / MP4 (MPEG-4 Part 14)'
OpenCV: FFMPEG: fallback to use tag 0x31637661/'avc1'


In [8]:
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1
    if frame_count % frame_interval != 0:  
        continue  

    results = model.predict(frame, conf=0.3, iou=0.4, classes=0, verbose=False)  

    count_in_polygon = 0  

    for result in results:
        boxes = result.boxes
        for box in boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cx, cy = (x1 + x2) // 2, (y1 + y2) // 2  

            if is_in_polygon(cx, cy):
                count_in_polygon += 1
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.circle(frame, (cx, cy), 5, (0, 0, 255), -1)

    frame_queue.append((count_in_polygon, time.time()))

    while len(frame_queue) > 0 and (time.time() - frame_queue[0][1] > average_duration):
        frame_queue.popleft()

    if len(frame_queue) > 0:
        average_count = np.mean([count for count, _ in frame_queue])
    else:
        average_count = 0

    cv2.polylines(frame, [np.array(polygon_points)], isClosed=True, color=(255, 0, 0), thickness=2)

    cv2.putText(frame, f'People Count: {count_in_polygon}', (10, 210), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
    cv2.putText(frame, f'Average Count: {average_count:.2f}', (10, 240), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
    cv2.putText(frame, f'Crowd Threshold: {crowd_threshold}', (10, 270), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
    cv2.putText(frame, f'Time Period: {average_duration}s', (10, 300), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

    if average_count >= crowd_threshold:
        cv2.putText(frame, 'Crowd Detected!', (10, 360), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)

    cv2.imshow('Crowd Detection', frame)
    videowriter.write(frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

videowriter.release()
cap.release()
cv2.destroyAllWindows()

In [ ]:
import cv2
import numpy as np

points = []
points_percent = []
screen = None

def get_coordinates(event, x, y, flags, param):
    global points, points_percent, screen
    if event == cv2.EVENT_LBUTTONDOWN:
        if screen is not None:
            h, w, c = screen.shape
            x_per, y_per = x / w, y / h
            points_percent.append([round(x_per,3), round(y_per,3)])
            points.append([x, y])
            print(f"Point captured: ({x}, {y})")
            print(points_percent)
            print(points)
        else:
            print("Screen is not set yet.")

def draw_polylines(frame, points):
    global screen
    screen = frame.copy()
    if len(points) > 1:
        cv2.polylines(frame, [np.array(points, dtype=np.int32)], isClosed=False, color=(0, 255, 0), thickness=2)
    for point in points:
        cv2.circle(frame, point, 5, (0, 0, 255), -1)

stream_url = './video_1.mp4'  # Replace with your video stream URL
cap = cv2.VideoCapture(stream_url)

cv2.namedWindow('Stream')
cv2.setMouseCallback('Stream', get_coordinates)

while True:
    ret, frame = cap.read()
    if not ret:
        print("Failed to grab frame")
        break

    frame = cv2.resize(frame, (1080, 720))
    draw_polylines(frame, points)
    cv2.imshow('Stream', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

print("Points:", points)
print("Points percent:", points_percent)

Point captured: (34, 36)
[[0.031, 0.05]]
[[34, 36]]
Point captured: (1040, 60)
[[0.031, 0.05], [0.963, 0.083]]
[[34, 36], [1040, 60]]
Point captured: (1048, 678)
[[0.031, 0.05], [0.963, 0.083], [0.97, 0.942]]
[[34, 36], [1040, 60], [1048, 678]]
Point captured: (38, 648)
[[0.031, 0.05], [0.963, 0.083], [0.97, 0.942], [0.035, 0.9]]
[[34, 36], [1040, 60], [1048, 678], [38, 648]]
Point captured: (32, 37)
[[0.031, 0.05], [0.963, 0.083], [0.97, 0.942], [0.035, 0.9], [0.03, 0.051]]
[[34, 36], [1040, 60], [1048, 678], [38, 648], [32, 37]]
Failed to grab frame
Points: [[34, 36], [1040, 60], [1048, 678], [38, 648], [32, 37]]
Points percent: [[0.031, 0.05], [0.963, 0.083], [0.97, 0.942], [0.035, 0.9], [0.03, 0.051]]
